# Finetuned model — inference & manual check

Run the finetuned Qwen2.5-3B LoRA on a transcript and inspect the extracted logistics CX metrics.

**Setup:** Run from repo root. Requires `unsloth`, `transformers`, etc. Model path: `models/qwen-logistics-lora/`.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))
import os
os.chdir(REPO_ROOT)  # so relative paths (e.g. models/) resolve

from inference import load_model, extract_with_finetuned

## Load model (run once)

Loads the finetuned LoRA from `models/qwen-logistics-lora/`. Takes ~30s on GPU.

In [ ]:
model, tokenizer = load_model()
print("Model loaded.")

## Your transcript

Paste or edit the conversation below, then run the next cell.

In [ ]:
transcript = """
Agent: Thank you for calling. How can I help?
Customer: My package says delivered but I never got it. I have a Ring camera and the driver didn't even stop.
Agent: I'm sorry to hear that. Can I get your tracking number?
Customer: 9400123456789.
Agent: I see it was marked delivered at 3 PM. I'll open an investigation and have someone contact you within 24 hours.
Customer: Okay, thanks.
"""

## Run extraction

In [ ]:
result = extract_with_finetuned(transcript, model=model, tokenizer=tokenizer)
print(result.model_dump_json(indent=2))

## Inspect fields (optional)

Pull out key fields for quick manual check.

In [ ]:
b = result.behavioral_analytics
o = result.operational_analytics
d = result.diagnostic_reasoning

print("Intent:", b.customer_intent.value)
print("Effort score (1-5):", b.customer_effort_score)
print("Sentiment trajectory:", b.sentiment_trajectory.value)
print("Quotes:", b.effort_and_friction_quotes[:2] if b.effort_and_friction_quotes else "(none)")
print("Resolution confirmed:", o.agent_explicitly_confirmed_resolution)
print("Unresolved next steps:", o.unresolved_next_steps)
print("Routing:", d.recommended_routing_queue)

## Batch test (optional)

Run on multiple transcripts from a list or file. Adjust `transcripts` as needed.

In [ ]:
# Example: list of short transcripts to test
transcripts = [
    transcript,  # reuse the one above
    """Agent: Hi, how can I help? Customer: Where's my order? Agent: Tracking number? Customer: 12345. Agent: It's in customs, we need the invoice. Customer: Ugh, fine.""",
]

for i, t in enumerate(transcripts):
    out = extract_with_finetuned(t, model=model, tokenizer=tokenizer, return_dict=True)
    intent = out["behavioral_analytics"]["customer_intent"]
    effort = out["behavioral_analytics"]["customer_effort_score"]
    print(f"--- Sample {i+1} --- Intent: {intent}, CES: {effort}")